In [ ]:
import pandas as pd
import os
from pathlib import Path

def load_raw_data(filepath):
    return pd.read_csv(filepath)

def clean_nav_dataset(dataframe):
    dataframe["date"] = pd.to_datetime(dataframe["date"], errors="coerce")
    
    dataframe = dataframe.sort_values(["amfi_code", "date"])
    
    dataframe["nav"] = dataframe.groupby("amfi_code")["nav"].ffill()
    
    dataframe = dataframe.drop_duplicates()
    dataframe = dataframe[dataframe["nav"] > 0]
    dataframe = dataframe.dropna(subset=["date"])
    
    return dataframe

def save_cleaned_data(dataframe, output_dir, filename):
    os.makedirs(output_dir, exist_ok=True)
    full_path = os.path.join(output_dir, filename)
    dataframe.to_csv(full_path, index=False)
    return full_path

def generate_report(original, cleaned):
    print(f"Original shape: {original.shape}")
    print(f"Cleaned shape: {cleaned.shape}")
    print(f"Rows removed: {original.shape[0] - cleaned.shape[0]}")
    print(f"\nMissing values:\n{cleaned.isnull().sum()}")
    print(f"\nDuplicate count: {cleaned.duplicated().sum()}")


INPUT_FILE = "data/raw/02_nav_history.csv"
OUTPUT_DIR = "data/processed"
OUTPUT_FILE = "clean_nav_history.csv"

raw_nav = load_raw_data(INPUT_FILE)

processed_nav = clean_nav_dataset(raw_nav)

generate_report(raw_nav, processed_nav)

saved_path = save_cleaned_data(processed_nav, OUTPUT_DIR, OUTPUT_FILE)
print(f"\nSaved: {saved_path}")

print(os.listdir(OUTPUT_DIR))

Original shape: (46000, 3)
Cleaned shape: (46000, 3)
Rows removed: 0

Missing values:
amfi_code    0
date         0
nav          0
dtype: int64

Duplicate count: 0

Saved: data/processed\clean_nav_history.csv
['clean_nav_history.csv']


In [17]:
import pandas as pd
import os

tx = pd.read_csv(
    "../data/raw/08_investor_transactions.csv"
)
print(tx.head())
tx.head()

  investor_id transaction_date  amfi_code transaction_type  amount_inr  \
0   INV003054       2024-01-01     119092              SIP        1834   
1   INV002952       2024-01-01     148567       Redemption      392882   
2   INV003420       2024-01-01     118636              SIP         912   
3   INV003436       2024-01-01     118634              SIP        1102   
4   INV004691       2024-01-01     119094          Lumpsum        8682   

         state       city city_tier age_group  gender  annual_income_lakh  \
0    Telangana  Hyderabad       T30       56+  Female                77.1   
1       Punjab   Amritsar       B30     18-25    Male                 7.1   
2      Haryana  Faridabad       B30     36-45    Male                47.2   
3  Maharashtra     Mumbai       T30     36-45  Female                54.4   
4        Delhi      Noida       T30     26-35    Male                14.5   

  payment_mode kyc_status  
0          UPI   Verified  
1       Cheque   Verified  
2      M

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [18]:
print(tx.dtypes)
print()

print(tx.isnull().sum())
print()

print(tx.duplicated().sum())

investor_id            object
transaction_date       object
amfi_code               int64
transaction_type       object
amount_inr              int64
state                  object
city                   object
city_tier              object
age_group              object
gender                 object
annual_income_lakh    float64
payment_mode           object
kyc_status             object
dtype: object

investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0
dtype: int64

0


In [20]:
tx["transaction_date"] = pd.to_datetime(
    tx["transaction_date"], errors="coerce"
)

tx.head()

print(tx["transaction_type"].unique())

['SIP' 'Redemption' 'Lumpsum']


In [21]:
tx["transaction_type"] = (
tx["transaction_type"]
.astype(str)
.str.strip()
.str.title()
)

print(
    tx["transaction_type"].unique()
)

['Sip' 'Redemption' 'Lumpsum']


In [22]:
print(
    tx["amount_inr"].describe()
)

count     32778.000000
mean     107437.318628
std      150415.905084
min         400.000000
25%        3153.000000
50%       17782.500000
75%      189324.250000
max      597498.000000
Name: amount_inr, dtype: float64


In [23]:
tx=tx[
    tx["amount_inr"] > 0
]

print(tx.shape)

(32778, 13)


In [25]:
print(
    tx["kyc_status"].unique()
)

['Verified' 'Pending']


In [27]:
print(
tx.duplicated().sum()
)

0


In [28]:
print(
    tx.isnull().sum()
)

investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0
dtype: int64


In [29]:
tx.to_csv("../data/processed/clean_investor_transactions.csv",
    index=False
    )
print("Ok")

Ok


In [ ]:

tx=pd.read_csv("../data/raw/07_scheme_performance.csv")

print(tx.shape)
tx.head()


(40, 19)


,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [ ]:
print(tx.dtypes)
print()

print(tx.isnull().sum())
print()

print(tx.duplicated().sum())



amfi_code               int64
scheme_name            object
fund_house             object
category               object
plan                   object
return_1yr_pct        float64
return_3yr_pct        float64
return_5yr_pct        float64
benchmark_3yr_pct     float64
alpha                 float64
beta                  float64
sharpe_ratio          float64
sortino_ratio         float64
std_dev_ann_pct       float64
max_drawdown_pct      float64
aum_crore               int64
expense_ratio_pct     float64
morningstar_rating      int64
risk_grade             object
dtype: object

amfi_code             0
scheme_name           0
fund_house            0
category              0
plan                  0
return_1yr_pct        0
return_3yr_pct        0
return_5yr_pct        0
benchmark_3yr_pct     0
alpha                 0
beta                  0
sharpe_ratio          0
sortino_ratio         0
std_dev_ann_pct       0
max_drawdown_pct      0
aum_crore             0
expense_ratio_pct     0
morning

In [48]:
print(
    tx.columns
)

Index(['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan',
       'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct',
       'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio',
       'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct',
       'morningstar_rating', 'risk_grade'],
      dtype='object')


In [50]:
cols = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct"
]
for c in cols:
    tx[c] = pd.to_numeric(tx[c],
    errors="coerce"
    )
print("ok")

ok


In [51]:
tx["expense_flag"] = (
    (tx["expense_ratio_pct"] < 1.0) 
|
(
    tx["expense_ratio_pct"] > 2.5)
)

print(
    tx["expense_flag"]
    .value_counts()
)

expense_flag
False    26
True     14
Name: count, dtype: int64


In [53]:
tx.to_csv(
    "../data/processed/clean_scheme_performance.csv",
    index=False
)
print("Ok")

Ok


In [54]:
print(
    os.listdir("../data/processed")
)

['clean_investor_transactions.csv', 'clean_nav_history.csv', 'clean_scheme_performance.csv']
